# 08 · Treino — Previsão de Volume de Incidentes (D+1 / D+7)

Treina **6 modelos** de regressão (3 tabelas de segmentação × 2
horizontes) e só considera um modelo "vencedor" se ele **bater uma
baseline ingênua** na validação — senão, a baseline é o resultado
recomendado pra esse segmento, documentado como tal (não é vergonha,
é honestidade de engenharia).

**Baseline**: `media_movel_7d` como previsão (já existe na tabela, não
precisa calcular nada a mais). Serve de piso mínimo de decência — um
modelo que não bate isso não está aprendendo nada útil.

**Busca de hiperparâmetro pequena** (3 combinações de
`maxDepth`/`maxIter`), escolhida **sempre pela validação**, nunca pelo
teste — o teste só é usado uma vez, no final, pra reportar o resultado
do candidato já escolhido (modelo ou baseline).

**Sem MLflow nesta primeira versão** (decisão explícita — tracking
formal fica pra depois, quando fizer sentido trazer o Model Registry
integrado ao Unity Catalog).

In [ ]:
%run ./00_config

In [ ]:
%run ./07_ml_prep

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [ ]:
def _mae_rmse(df, coluna_target: str, coluna_previsao: str):
    ev_mae = RegressionEvaluator(labelCol=coluna_target, predictionCol=coluna_previsao, metricName="mae")
    ev_rmse = RegressionEvaluator(labelCol=coluna_target, predictionCol=coluna_previsao, metricName="rmse")
    return ev_mae.evaluate(df), ev_rmse.evaluate(df)


print("_mae_rmse() pronta.")

## Função de treino com portão automático (baseline + tuning)

**Refeito com `Pipeline` do Spark ML** (`StringIndexer` +
`VectorAssembler` + `GBTRegressor` ajustados juntos, de uma vez só) —
corrige um problema de design da primeira versão: antes, o
`StringIndexer` era ajustado "na hora" dentro desta função e não ficava
guardado em lugar nenhum, então o notebook `09` (inferência) precisava
reaproveitar o `teste_df` já pré-processado — só que esse `teste_df`
tinha passado por um `dropna` de alvo pensado pro *treino*, o que
silenciosamente cortava a última linha real disponível (o dia mais
recente não tem `target_d1`/`target_d7` preenchido, por construção —
ninguém sabe "o dia seguinte" dele). Resultado: a previsão saía pro dia
errado. Com o `Pipeline`, a inferência vira só
`pipeline_ajustado.transform(linha_nova)` — sem depender de nenhum
pedaço do processo de treino além do modelo em si.

**Por isso o split agora acontece ANTES do `dropna`** de alvo — assim
o dia mais recente (sem `target` preenchido) continua disponível no
conjunto "bruto" de teste, pronto pra virar a linha de inferência real
no notebook `09`. O `dropna` só é aplicado nas cópias usadas pra
treinar/avaliar (que precisam de alvo preenchido).

`maxBins` calculado dinamicamente a partir da cardinalidade real da
coluna categórica (achado do teste original: `produto` tem ~47
categorias, `categoria` ~141+, o default de 32 do `GBTRegressor` não
cobre nenhum dos dois).

In [ ]:
CANDIDATOS_HIPERPARAMETRO = [
    {"maxDepth": 3, "maxIter": 30},
    {"maxDepth": 5, "maxIter": 50},
    {"maxDepth": 7, "maxIter": 80},
]


def treinar_com_gate(nome_tabela: str, coluna_target: str, coluna_segmento_categorica: str = None) -> dict:
    """
    Treina até 3 candidatos de GBTRegressor (variando maxDepth/maxIter, cada
    um dentro de um Pipeline com StringIndexer+VectorAssembler), escolhe o
    melhor pela validação, e só o declara "vencedor" se bater a baseline
    (media_movel_7d) na validação. Reporta a métrica final de teste do que
    venceu — modelo ou baseline, nunca os dois misturados.
    """
    colunas_lag = ["lag_1d", "lag_7d", "lag_14d", "media_movel_7d", "media_movel_14d"]

    df = carregar_features_com_calendario(nome_tabela).withColumn(
        "is_fim_de_semana_int", F.col("is_fim_de_semana").cast("int")
    )

    # Split ANTES do dropna — o dia mais recente (sem target, por construção)
    # precisa sobreviver no conjunto "bruto" de teste para a inferência real
    # do notebook 09. Only as cópias "_treino"/"_validacao"/"_teste" (com
    # dropna) são usadas para ajustar/avaliar o modelo.
    treino_bruto, validacao_bruto, teste_bruto = split_temporal(df)

    treino = treino_bruto.dropna(subset=colunas_lag + [coluna_target])
    validacao = validacao_bruto.dropna(subset=colunas_lag + [coluna_target])
    teste = teste_bruto.dropna(subset=colunas_lag + [coluna_target])

    colunas_numericas = list(colunas_lag) + [
        "dia_semana_num", "trimestre", "is_feriado",
        "qtd_kpi_regra_divergente", "is_fim_de_semana_int",
    ]
    if coluna_segmento_categorica != "prioridade_num":
        colunas_numericas.append("prioridade_num")

    stages = []
    colunas_features = list(colunas_numericas)
    n_categorias = 0
    if coluna_segmento_categorica:
        indexer = StringIndexer(
            inputCol=coluna_segmento_categorica,
            outputCol=f"{coluna_segmento_categorica}_idx",
            handleInvalid="keep",
        )
        stages.append(indexer)
        colunas_features.append(f"{coluna_segmento_categorica}_idx")
        n_categorias = df.select(coluna_segmento_categorica).distinct().count()

    assembler = VectorAssembler(inputCols=colunas_features, outputCol="features")
    stages.append(assembler)
    max_bins = max(32, n_categorias + 5)

    mae_baseline_val, _ = _mae_rmse(validacao, coluna_target, "media_movel_7d")

    melhor = None
    for cand in CANDIDATOS_HIPERPARAMETRO:
        gbt = GBTRegressor(
            featuresCol="features", labelCol=coluna_target,
            maxIter=cand["maxIter"], maxDepth=cand["maxDepth"], maxBins=max_bins, seed=42,
        )
        pipeline = Pipeline(stages=stages + [gbt])
        pipeline_ajustado = pipeline.fit(treino)
        mae_val, rmse_val = _mae_rmse(pipeline_ajustado.transform(validacao), coluna_target, "prediction")
        if melhor is None or mae_val < melhor["mae_val"]:
            melhor = {**cand, "mae_val": mae_val, "rmse_val": rmse_val, "pipeline": pipeline_ajustado}

    vencedor = "modelo" if melhor["mae_val"] < mae_baseline_val else "baseline"

    if vencedor == "modelo":
        mae_teste, rmse_teste = _mae_rmse(melhor["pipeline"].transform(teste), coluna_target, "prediction")
    else:
        mae_teste, rmse_teste = _mae_rmse(teste, coluna_target, "media_movel_7d")

    return {
        "vencedor": vencedor,
        "hiperparametros": f"maxDepth={melhor['maxDepth']},maxIter={melhor['maxIter']}" if vencedor == "modelo" else "-",
        "mae_val_modelo": round(melhor["mae_val"], 3),
        "mae_val_baseline": round(mae_baseline_val, 3),
        "mae_teste_final": round(mae_teste, 3),
        "rmse_teste_final": round(rmse_teste, 3),
        "n_treino": treino.count(),
        "n_validacao": validacao.count(),
        "n_teste": teste.count(),
        # Pipeline inteiro (indexação + montagem de vetor + modelo) — a
        # inferência no notebook 09 só precisa chamar .transform() nele,
        # direto na linha nova, sem depender de mais nada do treino.
        "pipeline_ajustado": melhor["pipeline"] if vencedor == "modelo" else None,
        # Conjunto de teste SEM o dropna de alvo — inclui o dia mais recente
        # de verdade, é daqui que o notebook 09 pega a linha de inferência.
        "teste_bruto": teste_bruto,
        "coluna_segmento_categorica": coluna_segmento_categorica,
    }


print("treinar_com_gate() pronta.")

## Treinar as 6 combinações

In [ ]:
configuracoes = [
    ("features_series_produto", "produto"),
    ("features_series_categoria", "categoria"),
    ("features_series_prioridade", None),
]
horizontes = ["target_d1", "target_d7"]

resultados_por_chave = {}
resumo_resultados = []

for nome_tabela, coluna_segmento in configuracoes:
    for horizonte in horizontes:
        chave = f"{nome_tabela}__{horizonte}"
        print(f"Treinando: {chave}")
        resultado = treinar_com_gate(nome_tabela, horizonte, coluna_segmento)
        resultados_por_chave[chave] = resultado
        resumo_resultados.append({
            "tabela": nome_tabela, "horizonte": horizonte, "vencedor": resultado["vencedor"],
            "hiperparametros": resultado["hiperparametros"],
            "mae_val_modelo": resultado["mae_val_modelo"], "mae_val_baseline": resultado["mae_val_baseline"],
            "mae_teste_final": resultado["mae_teste_final"], "rmse_teste_final": resultado["rmse_teste_final"],
        })

print(f"\n{len(resultados_por_chave)} combinações avaliadas.")

## Resumo comparativo

⚠️ **Sobre segmentos com poucos dados de validação** (ex.:
`features_series_prioridade`, só 3 segmentos): um modelo pode vencer a
baseline na validação e ainda assim performar pior no teste — a
validação tem poucas linhas nesses casos, então "vencer" ali pode ser
ruído estatístico, não sinal real. Isso não invalida o portão (é o
critério mais honesto que temos), mas é um limite conhecido a reportar
pra banca, não esconder.

In [ ]:
resumo_df = spark.createDataFrame(resumo_resultados)
display(resumo_df.select(
    "tabela", "horizonte", "vencedor", "hiperparametros",
    "mae_val_modelo", "mae_val_baseline", "mae_teste_final", "rmse_teste_final",
))

qtd_modelo = sum(1 for r in resumo_resultados if r["vencedor"] == "modelo")
qtd_baseline = sum(1 for r in resumo_resultados if r["vencedor"] == "baseline")
print(f"\nModelo venceu em {qtd_modelo} de {len(resumo_resultados)} combinações.")
print(f"Baseline venceu em {qtd_baseline} de {len(resumo_resultados)} combinações — recomendada nesses casos, não é falha do projeto.")